In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l1.5 Gradients and autograd

> 🎯 **Goal:** See how PyTorch computes gradients automatically, the magic that lets a model improve itself without you doing any calculus.

**Why this matters.** In the last lesson we got one number, the loss, that says how wrong the model is. But knowing you are wrong is not enough; you need to know which way to adjust each parameter to become less wrong. That direction is the **gradient**. PyTorch's **autograd** (automatic gradient) engine computes the gradient for every parameter automatically, which is the single feature that makes training large models practical.

**The intuition.** Imagine standing on a foggy hillside wanting to reach the valley (low loss). You cannot see far, but you can feel the slope under your feet. The gradient is exactly that slope: it tells you which direction is uphill and how steep. To go down, you step opposite the gradient. A neural network has millions of parameters, so the "hillside" has millions of dimensions, but the idea is identical: the gradient points uphill, and learning means stepping the other way.

**The mechanics.** A gradient is the derivative of the loss with respect to a parameter: how much the loss changes if you nudge that parameter a little. Three ingredients make autograd work:

- **`requires_grad=True`** marks a tensor as something we want gradients for. PyTorch then secretly records every operation done to it, building a **computation graph** (a record of what was computed from what).
- **`.backward()`** runs the chain rule backward through that graph, computing the gradient of the final value with respect to every tracked tensor.
- **`.grad`** is where the result lands, attached to each tracked tensor.

In the code, `y = (x ** 2).sum()`. From calculus, the derivative of `x**2` is `2*x`, so at `x = 3` the gradient should be `2 * 3 = 6`. We never typed that derivative; autograd figures it out.

In [ ]:
x = torch.tensor([3.0], requires_grad=True)
y = (x ** 2).sum()   # dy/dx = 2x, so at x=3 the gradient is 6
y.backward()
print("x.grad:", x.grad)

**What just happened.** `x.grad` printed `[6.]`, exactly the `2*x` derivative evaluated at `x = 3`, computed for us without writing a single line of calculus. That is autograd: you describe the forward computation, and it derives the backward gradients. Every training step in this entire course follows the same four-beat rhythm: forward (compute predictions), loss (score them), `backward()` (get gradients), optimizer step (nudge parameters opposite the gradient), then repeat.

⚠️ **Common pitfalls**
- Forgetting `requires_grad=True`. Without it PyTorch tracks nothing, so `.grad` stays `None` and the model can never learn.
- Gradients accumulate. Calling `.backward()` again adds to the existing `.grad` instead of replacing it. That is why real training loops call `optimizer.zero_grad()` before each backward pass; forgetting it makes gradients pile up and training go haywire.

🏋️ **Try it yourself.** Change the starting value to `torch.tensor([5.0], requires_grad=True)` and rerun. The gradient should print `[10.]`, because `2*x` at `x = 5` is `10`. Try a few values and confirm the printed gradient is always exactly twice the input. You have just verified autograd against hand calculus.

In [ ]:
assert torch.allclose(x.grad, torch.tensor([6.0]))